# Logistic regression without using the conjunctive rules feature

## Imports

In [1]:
import mlflow
import pandas as pd

from heart_failure.dataset import split_data
from heart_failure.modeling.train import optuna_search_lr
from heart_failure.config.modeling import CV_SPLITS, LR_N_TRIALS, LR_SCORING, F_BETA_THRESHOLD
from heart_failure.config.config import INTERIM_DATA_DIR, HEART_DISEASE, SEX, PROCESSED_DATA_DIR
from heart_failure.config.features import TEST_SIZE, RANDOM_STATE, VAL_SIZE
from heart_failure.reports import print_best_params
from heart_failure.modeling.evaluate import (
    get_metrics, log_metrics, find_best_threshold
)
from heart_failure.modeling.predict import predict_by_threshold

2026-06-18 13:43:59.686 | INFO     | heart_failure.config.config:<module>:11 - PROJ_ROOT path is: D:\heart_failure


In [2]:
MODEL_NAME = "logistic_regression_without_conj_feature"
mlflow.set_experiment(f"heart_failure")

<Experiment: artifact_location='file:D:/heart_failure/notebooks/mlruns/1', creation_time=1781534399229, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781534399229, lifecycle_stage='active', name='heart_failure', tags={}, trace_location=None, workspace='default'>

In [3]:
df = pd.read_csv(INTERIM_DATA_DIR / "heart.csv")
rem_columns = pd.read_csv(PROCESSED_DATA_DIR / "remaining_cols.csv")
rem_columns = rem_columns.squeeze().to_list()
rem_columns.remove("remainder__conjunctive_rules")

y = df[HEART_DISEASE]
X = df.drop(columns=[HEART_DISEASE])

X_train, y_train, X_val, y_val, X_test, y_test = split_data(
    X, y, stratify_func=lambda X, y: pd.concat([X[[SEX]], y], axis=1)
)

## Training

In [10]:
study, model = optuna_search_lr(
    X_train,
    y_train,
    n_trials=LR_N_TRIALS,
    n_splits=CV_SPLITS,
    scoring=LR_SCORING,
    rem_columns=rem_columns,
    random_state=RANDOM_STATE,
)

[I 2026-06-18 13:45:40,374] A new study created in memory with name: no-name-e8dd0313-e1a9-42b9-be05-62872f60f5e3
[I 2026-06-18 13:45:52,445] Trial 1 finished with value: 0.9266655118179259 and parameters: {'model__solver': 'liblinear', 'model__C': 0.0011088966450223342, 'model__max_iter': 425, 'model__tol': 0.0002307445389563467, 'model__l1_ratio': 0}. Best is trial 1 with value: 0.9266655118179259.
[I 2026-06-18 13:45:52,666] Trial 5 finished with value: 0.9236926957913296 and parameters: {'model__solver': 'liblinear', 'model__C': 3.8611045508582293, 'model__max_iter': 258, 'model__tol': 1.1458607702748293e-05, 'model__l1_ratio': 0}. Best is trial 1 with value: 0.9266655118179259.
[I 2026-06-18 13:45:52,723] Trial 10 finished with value: 0.8784054322653574 and parameters: {'model__solver': 'liblinear', 'model__C': 0.009356441632687784, 'model__max_iter': 358, 'model__tol': 0.0001677769221251018, 'model__l1_ratio': 1}. Best is trial 1 with value: 0.9266655118179259.
[I 2026-06-18 13:4

In [11]:
print_best_params(study)

Best parameters: {'model__solver': 'lbfgs', 'model__C': 0.024818485277676976, 'model__max_iter': 101, 'model__tol': 0.0001027891603614038}
Best score: 0.93040


In [12]:
lr = model.named_steps["model"]
weights = pd.Series(lr.coef_.ravel(), index=rem_columns)
weights.sort_values(key=abs, ascending=False)

target_encoder__ST_Slope          0.743823
target_encoder__ChestPainType     0.530269
binary_encoder__Sex               0.376774
binary_encoder__ExerciseAngina    0.361694
binary_encoder__FastingBS         0.347910
remainder__Oldpeak                0.287779
binarizer__Cholesterol            0.185030
binarizer__MaxHR                 -0.166455
remainder__MaxHR_z               -0.125722
binarizer__Age                    0.121415
dtype: float64

The cv score remained virtually unchanged (compared to the model with the existing feature).

### Threshold searching

In [13]:
y_train_proba = model.predict_proba(X_train)[:, 1]
best_threshold = find_best_threshold(y_train, y_train_proba, F_BETA_THRESHOLD)
best_threshold

np.float64(0.383)

## Scores

In [15]:
print("Validate metrics")
y_val_pred = predict_by_threshold(model, X_val, best_threshold)
y_val_proba = model.predict_proba(X_val)[:, 1]
val_metrics = get_metrics(y_val, y_val_pred, y_val_proba)
val_metrics

Validate metrics


precision    0.781818
recall       0.877551
f2_score     0.856574
pr_auc       0.920624
dtype: float64

The metrics are even slightly worse than those of a baseline. So, the feature is useful. Most likely, there are some nonlinear dependencies.

In [16]:
with mlflow.start_run(run_name=MODEL_NAME) as run:
    mlflow.log_param("model", MODEL_NAME)
    mlflow.log_param("test_size", TEST_SIZE)
    mlflow.log_param("val_size", VAL_SIZE)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_param("cv_splits", CV_SPLITS)
    mlflow.log_param("optuna_n_trials", LR_N_TRIALS)
    mlflow.log_param("remaining_columns", rem_columns)
    mlflow.log_param("f_beta_threshold", F_BETA_THRESHOLD)
    mlflow.log_param("proba_threshold", best_threshold)

    log_metrics(val_metrics, "val_")